<a href="https://colab.research.google.com/github/iagoxyz01/Prototipo/blob/main/Projeto_Big_Data_Gastos_Eleitorais_TSE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Big Data — Gastos eleitorais (TSE)

## Objetivo
Investigar se os candidatos que mais gastaram nas campanhas foram eleitos.

## Escopo
- **Presidente:** Brasil inteiro — 2018, 2022 e 2026.
- **Governador:** nove estados do Nordeste (AL, BA, CE, MA, PB, PE, PI, RN e SE) — 2018, 2022 e 2026.

## Tabelas que serão geradas
1. Todos os candidatos selecionados, com data, cargo, estado, nome, partido, gasto total e situação eleitoral.
2. Os 10 candidatos que mais gastaram em cada combinação de ano e cargo.
3. Categorias de despesas dos 10 maiores gastadores.

**Fonte:** arquivos oficiais do Tribunal Superior Eleitoral (TSE).

> Observação: os arquivos e nomes de colunas do TSE podem variar entre eleições. Este notebook tenta localizar os recursos automaticamente e interrompe o processamento do ano quando não encontra a estrutura esperada. Confira as mensagens de erro antes de interpretar os resultados.


## 1. Instalar/importar bibliotecas

In [ ]:
import io
import re
import zipfile
import unicodedata
from pathlib import Path

import pandas as pd
import requests

ANOS = [2018, 2022, 2026]
ESTADOS_NE = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]

DATAS = {
    2018: "07/10/2018",
    2022: "02/10/2022",
    2026: "04/10/2026"
}

API_TSE = "https://dadosabertos.tse.jus.br/api/3/action/package_show"

print("Bibliotecas e configurações carregadas.")

## 2. Funções para consultar e localizar arquivos no TSE

In [ ]:
def consultar_conjunto(slug):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(API_TSE, params={"id": slug}, headers=headers, timeout=45)
        resp.raise_for_status()
        dados = resp.json()
        if dados.get("success"):
            return dados["result"].get("resources", [])
        print("Dataset não encontrado:", slug)
    except Exception as exc:
        print(f"Consulta {slug} falhou: {type(exc).__name__}: {exc}")
    return None


def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    return "".join(ch for ch in texto if not unicodedata.combining(ch))


def escolher_recurso(recursos, palavras):
    if not recursos:
        return None
    for recurso in recursos:
        nome = normalizar(recurso.get("name", ""))
        if all(p in nome for p in palavras):
            return recurso
    return None


def localizar_recursos(ano):
    candidaturas = consultar_conjunto(f"candidatos-{ano}")
    contas = None
    for slug in [
        f"prestacao-de-contas-eleitorais-{ano}",
        f"prestacao-contas-{ano}",
        f"dadosabertos-tse-jus-br-dataset-prestacao-de-contas-eleitorais-{ano}",
    ]:
        contas = consultar_conjunto(slug)
        if contas is not None:
            break
    resultados = consultar_conjunto(f"resultados-{ano}")
    if resultados is None:
        resultados = consultar_conjunto(f"resultados-eleitorais-{ano}")
    if candidaturas is None or contas is None or resultados is None:
        raise ValueError(
            f"Catalogos TSE falharam para {ano}: "
            f"candidaturas={candidaturas is not None}, contas={contas is not None}, "
            f"resultados={resultados is not None}"
        )

    candidatos_res = (
        escolher_recurso(candidaturas, ["consulta", "cand", "brasil"])
        or escolher_recurso(candidaturas, ["consulta", "cand"])
        or escolher_recurso(candidaturas, ["candidatos"])
    )
    despesas_res = (
        escolher_recurso(contas, ["despesas", "candidatos"])
        or escolher_recurso(contas, ["candidatos"])
    )
    resultados_res = (
        escolher_recurso(resultados, ["votacao", "candidato", "brasil"])
        or escolher_recurso(resultados, ["votacao", "candidato"])
        or escolher_recurso(resultados, ["votacao", "nominal"])
    )
    recursos = {"candidatos": candidatos_res, "despesas": despesas_res, "resultados": resultados_res}
    faltantes = [k for k, v in recursos.items() if v is None]
    if faltantes:
        raise ValueError(f"Recursos não encontrados ({faltantes}); candidatos disponíveis: "
                         f"{[r.get('name') for r in candidaturas]}; despesas disponíveis: "
                         f"{[r.get('name') for r in contas]}; resultados disponíveis: "
                         f"{[r.get('name') for r in resultados]}")
    return recursos


## 3. Baixar e ler os arquivos

In [ ]:
import time


def baixar_conteudo(recurso, max_tentativas=3, intervalo=5):
    print("Baixando:", recurso.get("name"))
    erro_final = None
    for tentativa in range(1, max_tentativas + 1):
        try:
            response = requests.get(recurso["url"], timeout=300, stream=True)
            response.raise_for_status()
            buffer = io.BytesIO()
            for bloco in response.iter_content(chunk_size=1024 * 1024):
                if bloco:
                    buffer.write(bloco)
            return buffer.getvalue()
        except Exception as exc:
            erro_final = exc
            print(f"Tentativa {tentativa}/{max_tentativas}: {exc}")
            if tentativa < max_tentativas:
                time.sleep(intervalo)
    raise ConnectionError(f"Download falhou: {erro_final}")


def escolher_nome_csv_zip(arquivo_zip, palavras=(), tipo=None):
    nomes = [n for n in arquivo_zip.namelist() if n.lower().endswith((".csv", ".txt"))]
    if not nomes:
        raise ValueError("ZIP não contém CSV/TXT.")
    norm = {n: normalizar(n) for n in nomes}
    filtrados = [n for n in nomes if all(p in norm[n] for p in palavras)]
    if tipo == "despesas":
        selecionados = [n for n in nomes if "despesa" in norm[n]]
    elif tipo == "resultados":
        selecionados = [n for n in nomes if "votacao" in norm[n] and ("candidato" in norm[n] or "nominal" in norm[n])]
    elif tipo == "candidatos":
        selecionados = [n for n in nomes if "cand" in norm[n]]
    else:
        selecionados = filtrados or nomes
    selecionados = selecionados or filtrados or nomes

    def nacional(nome):
        x = norm[nome].replace("\\", "/")
        return "brasil" in x or x.startswith("br_") or "/br_" in x or "_br_" in x
    nacionais = [n for n in selecionados if nacional(n)]
    selecionados = nacionais or selecionados
    selecionados.sort(key=lambda n: (0 if "brasil" in norm[n] else 1, len(n)))
    return selecionados[0]


def ler_tabela(conteudo, nome_arquivo=None, nrows=None):
    ultimo_erro = None
    for enc in ("latin1", "utf-8-sig", "utf-8"):
        try:
            return pd.read_csv(io.BytesIO(conteudo), sep=None, engine="python",
                               encoding=enc, nrows=nrows)
        except Exception as exc:
            ultimo_erro = exc
    raise ValueError(f"CSV não foi lido ({nome_arquivo}): {ultimo_erro}")


def baixar_e_ler(recurso, palavras_internas=()):
    conteudo = baixar_conteudo(recurso)
    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nome = escolher_nome_csv_zip(z, palavras_internas)
            with z.open(nome) as stream:
                return ler_tabela(stream.read(), nome)
    return ler_tabela(conteudo, recurso.get("name"))


## 4. Funções para identificar colunas e somar despesas

In [ ]:
def encontrar_coluna(df, alternativas):
    for nome in alternativas:
        if nome in df.columns:
            return nome
    raise ValueError(
        f"Coluna não encontrada. Esperadas: {alternativas}. "
        f"Colunas disponíveis: {list(df.columns)}"
    )


def converter_valores_monetarios(serie):
    texto = serie.astype(str).str.strip().str.replace("R$", "", regex=False).str.strip()
    texto = texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce").fillna(0)


def criar_mapa_prestadores(candidatos, id_col):
    """Mapeia SQ_PRESTADOR_CONTAS para SQ_CANDIDATO usando arquivo de candidatos."""
    prestador_col = encontrar_coluna(candidatos, ["SQ_PRESTADOR_CONTAS"])
    base = candidatos[[prestador_col, id_col]].dropna().copy()
    base[prestador_col] = base[prestador_col].astype(str).str.strip()
    base[id_col] = base[id_col].astype(str).str.strip()
    base = base.drop_duplicates(subset=[prestador_col])
    return dict(zip(base[prestador_col], base[id_col]))


def somar_despesas(recurso, mapa_prestadores, ids_candidatos):
    """Soma VR_PAGTO_DESPESA cruzando SQ_PRESTADOR_CONTAS com candidato."""
    conteudo = baixar_conteudo(recurso)
    totais = {}
    ids_candidatos = set(map(str, ids_candidatos))
    prestadores_validos = {
        str(prestador): str(candidato)
        for prestador, candidato in mapa_prestadores.items()
        if str(candidato) in ids_candidatos
    }

    def processar(stream):
        for chunk in pd.read_csv(
            stream, sep=";", encoding="latin1", chunksize=250_000, low_memory=False
        ):
            prestador_col = encontrar_coluna(chunk, ["SQ_PRESTADOR_CONTAS"])
            valor_col = encontrar_coluna(chunk, ["VR_PAGTO_DESPESA", "VR_DESPESA_CONTRATADA", "VR_DESPESA"])
            chunk[prestador_col] = chunk[prestador_col].astype(str).str.strip()
            chunk = chunk[chunk[prestador_col].isin(prestadores_validos)].copy()
            if chunk.empty:
                continue
            chunk[valor_col] = converter_valores_monetarios(chunk[valor_col])
            chunk["_id_candidato"] = chunk[prestador_col].map(prestadores_validos)
            agrupado = chunk.groupby("_id_candidato")[valor_col].sum()
            for candidato, valor in agrupado.items():
                totais[str(candidato)] = totais.get(str(candidato), 0) + float(valor)

    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nome = escolher_nome_csv_zip(z, tipo="despesas")
            print("Arquivo de despesas selecionado:", nome)
            with z.open(nome) as stream:
                processar(stream)
    else:
        processar(io.BytesIO(conteudo))
    return totais


## 5. Identificar a situação eleitoral

In [ ]:
def identificar_eleitos(recurso, ids_candidatos):
    """Identifica candidatos com situação final contendo ELEITO processando todos os CSVs necessários de votação."""
    conteudo = baixar_conteudo(recurso)
    eleitos = set()

    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as arquivo_zip:
            # Procuramos primeiro o arquivo consolidado Brasil para poupar processamento
            nomes_csv = [nome for nome in arquivo_zip.namelist() if nome.lower().endswith((".csv", ".txt"))]

            # Se houver um arquivo unificado para o BRASIL, usamos ele
            brasil_csv = [n for n in nomes_csv if "brasil" in n.lower() or "_br" in n.lower()]
            arquivos_para_processar = brasil_csv if brasil_csv else nomes_csv

            print(f"Processando {len(arquivos_para_processar)} arquivo(s) de resultado no ZIP...")
            for nome_csv in arquivos_para_processar:
                # Apenas processa se for arquivo de votação nominal de candidato por município/zona ou similar
                if "votacao_candidato" in nome_csv.lower() or "votacao_nominal" in nome_csv.lower() or brasil_csv:
                    conteudo_csv = arquivo_zip.read(nome_csv)

                    for parte in pd.read_csv(
                        io.BytesIO(conteudo_csv),
                        sep=";",
                        encoding="latin1",
                        chunksize=250_000,
                        low_memory=False
                    ):
                        id_col = encontrar_coluna(parte, ["SQ_CANDIDATO"])
                        situacao_col = encontrar_coluna(parte, ["DS_SIT_TOT_TURNO", "DS_SITUACAO_TSE"])

                        parte[id_col] = parte[id_col].astype(str).str.strip()
                        parte = parte[parte[id_col].isin(ids_candidatos)]

                        situacao = parte[situacao_col].fillna("").astype(str).str.upper()
                        filtro_eleito = (
                            situacao.str.contains("ELEITO")
                            & ~situacao.str.contains("NAO ELEITO")
                            & ~situacao.str.contains("NÃO ELEITO")
                        )
                        eleitos.update(parte.loc[filtro_eleito, id_col].unique())
    else:
        for parte in pd.read_csv(
            io.BytesIO(conteudo),
            sep=";",
            encoding="latin1",
            chunksize=250_000,
            low_memory=False
        ):
            id_col = encontrar_coluna(parte, ["SQ_CANDIDATO"])
            situacao_col = encontrar_coluna(parte, ["DS_SIT_TOT_TURNO", "DS_SITUACAO_TSE"])

            parte[id_col] = parte[id_col].astype(str).str.strip()
            parte = parte[parte[id_col].isin(ids_candidatos)]

            situacao = parte[situacao_col].fillna("").astype(str).str.upper()
            filtro_eleito = (
                situacao.str.contains("ELEITO")
                & ~situacao.str.contains("NAO ELEITO")
                & ~situacao.str.contains("NÃO ELEITO")
            )
            eleitos.update(parte.loc[filtro_eleito, id_col].unique())

    return eleitos

## 6. Processar eleições de 2018, 2022 e 2026

In [ ]:
registros = []
erros = []
mapa_prestadores_por_ano = {}
candidatos_por_ano = {}

for ano in ANOS:
    print("\\n" + "=" * 65)
    print("PROCESSANDO:", ano)
    try:
        recursos = localizar_recursos(ano)
        candidatos = baixar_e_ler(recursos["candidatos"], palavras_internas=[])
        id_col = encontrar_coluna(candidatos, ["SQ_CANDIDATO"])
        nome_col = encontrar_coluna(candidatos, ["NM_URNA_CANDIDATO", "NM_CANDIDATO"])
        partido_col = encontrar_col(candidatos, ["SG_PARTIDO"])
        cargo_col = encontrar_col(candidatos, ["DS_CARGO"])
        uf_col = encontrar_col(candidatos, ["SG_UF"])
        candidatos[id_col] = candidatos[id_col].astype(str).str.strip()

        presidentes = candidatos[candidatos[cargo_col].astype(str).str.upper() == "PRESIDENTE"].copy()
        governadores = candidatos[
            (candidatos[cargo_col].astype(str).str.upper() == "GOVERNADOR")
            & (candidatos[uf_col].isin(ESTADOS_NE))
        ].copy()
        selecionados = pd.concat([presidentes, governadores], ignore_index=True)
        selecionados = selecionados.drop_duplicates(subset=[id_col]).reset_index(drop=True)
        ids = set(selecionados[id_col].dropna().astype(str))
        if not ids:
            raise ValueError("Nenhum presidente ou governador do Nordeste encontrado.")

        # Cruzamento oficial pela chave do prestador, se ela estiver no cadastro carregado.
        mapa_prestadores = criar_mapa_prestadores(candidatos, id_col)
        mapa_prestadores_por_ano[ano] = mapa_prestadores
        candidatos_por_ano[ano] = candidatos
        despesas = somar_despesas(recursos["despesas"], mapa_prestadores, ids)
        eleitos = identificar_eleitos(recursos["resultados"], ids)

        for _, candidato in selecionados.iterrows():
            identificador = str(candidato[id_col])
            registros.append({
                "Data da eleição": DATAS[ano],
                "Ano": ano,
                "Cargo": candidato[cargo_col],
                "Estado": candidato[uf_col],
                "SQ_CANDIDATO": identificador,
                "Candidato": candidato[nome_col],
                "Partido": candidato[partido_col],
                "Gasto total (R$)": despesas.get(identificador, 0),
                "Eleito": "Sim" if identificador in eleitos else "Não",
            })
        print(f"Candidatos selecionados: {len(selecionados)}")
        print(f"Candidatos com despesas identificadas: {sum(1 for i in ids if despesas.get(i, 0) > 0)}")
        print("Ano processado.")
    except Exception as erro:
        erros.append({"Ano": ano, "Erro": f"{type(erro).__name__}: {erro}"})
        print("ERRO:", erros[-1]["Erro"])

print("\\nResumo dos erros:")
display(pd.DataFrame(erros))


## 7. Tabela geral de candidatos

In [ ]:
if erros:
    print("ATENÇÃO — anos com falha; os resultados abaixo são parciais:")
    display(pd.DataFrame(erros))

colunas_resultado = [
    "Data da eleição", "Ano", "Cargo", "Estado", "SQ_CANDIDATO",
    "Candidato", "Partido", "Gasto total (R$)", "Eleito"
]
tabela_geral = pd.DataFrame(registros, columns=colunas_resultado)
if tabela_geral.empty:
    print("Nenhum ano foi processado com sucesso. Não há dados para exibir.")
else:
    tabela_geral["Gasto total (R$)"] = pd.to_numeric(
        tabela_geral["Gasto total (R$)"], errors="coerce"
    ).fillna(0)
    tabela_geral = tabela_geral.drop_duplicates(
        subset=["Ano", "SQ_CANDIDATO"], keep="first"
    )
    tabela_geral = tabela_geral.sort_values(
        ["Ano", "Cargo", "Gasto total (R$)"],
        ascending=[True, True, False]
    ).reset_index(drop=True)
    display(tabela_geral.style.format({"Gasto total (R$)": "R$ {:,.2f}"}))


## 8. Os 10 candidatos que mais gastaram por ano e cargo


In [ ]:
if tabela_geral.empty:
    top10 = pd.DataFrame(columns=tabela_geral.columns)
    print("Top 10 indisponível: nenhum candidato foi processado.")
else:
    top10 = (
        tabela_geral.sort_values("Gasto total (R$)", ascending=False)
        .groupby(["Ano", "Cargo"], group_keys=False)
        .head(10)
        .reset_index(drop=True)
    )
    display(top10.style.format({"Gasto total (R$)": "R$ {:,.2f}"}))


## 9. Onde foram gastos os recursos dos 10 maiores gastadores


In [ ]:
# Detalhamento das despesas por categoria dos candidatos selecionados no Top 10.
detalhes_categoria = []
if not top10.empty:
    for ano in sorted(top10["Ano"].unique()):
        ids_top = set(top10.loc[top10["Ano"] == ano, "SQ_CANDIDATO"].astype(str))
        nomes_top = top10.loc[top10["Ano"] == ano, ["SQ_CANDIDATO", "Candidato", "Ano"]].drop_duplicates()
        mapa = mapa_prestadores_por_ano.get(int(ano), {})
        mapa_invertido = {str(cid): str(prest) for prest, cid in mapa.items() if str(cid) in ids_top}
        recurso = localizar_recursos(int(ano))["despesas"]
        conteudo_zip = baixar_conteudo(recurso)

        def acumular_categoria(stream):
            for chunk in pd.read_csv(stream, sep=";", encoding="latin1", chunksize=250_000, low_memory=False):
                prestador_col = encontrar_coluna(chunk, ["SQ_PRESTADOR_CONTAS"])
                categoria_col = encontrar_coluna(chunk, ["DS_ORIGEM_DESPESA", "DS_DESPESA", "DS_TIPO_DESPESA"])
                valor_col = encontrar_coluna(chunk, ["VR_PAGTO_DESPESA", "VR_DESPESA_CONTRATADA", "VR_DESPESA"])
                chunk[prestador_col] = chunk[prestador_col].astype(str).str.strip()
                chunk = chunk[chunk[prestador_col].isin(mapa_invertido)].copy()
                if chunk.empty:
                    continue
                chunk[valor_col] = converter_valores_monetarios(chunk[valor_col])
                chunk[categoria_col] = chunk[categoria_col].fillna("Categoria não informada").astype(str)
                chunk["SQ_CANDIDATO"] = chunk[prestador_col].map(mapa_invertido)
                soma = chunk.groupby(["SQ_CANDIDATO", categoria_col], as_index=False)[valor_col].sum()
                soma = soma.merge(nomes_top, on="SQ_CANDIDATO", how="left")
                soma = soma.rename(columns={categoria_col: "Categoria de despesa", valor_col: "Valor gasto (R$)"})
                detalhes_categoria.append(soma[["Ano", "Candidato", "Categoria de despesa", "Valor gasto (R$)"]])

        if zipfile.is_zipfile(io.BytesIO(conteudo_zip)):
            with zipfile.ZipFile(io.BytesIO(conteudo_zip)) as z:
                nome_csv = escolher_nome_csv_zip(z, tipo="despesas")
                with z.open(nome_csv) as stream:
                    acumular_categoria(stream)
        else:
            acumular_categoria(io.BytesIO(conteudo_zip))

if detalhes_categoria:
    gastos_por_categoria = pd.concat(detalhes_categoria, ignore_index=True)
    gastos_por_categoria = gastos_por_categoria.groupby(
        ["Ano", "Candidato", "Categoria de despesa"], as_index=False
    )["Valor gasto (R$)"].sum().sort_values(
        ["Ano", "Candidato", "Valor gasto (R$)"], ascending=[True, True, False]
    )
    display(gastos_por_categoria.style.format({"Valor gasto (R$)": "R$ {:,.2f}"}))
else:
    gastos_por_categoria = pd.DataFrame(
        columns=["Ano", "Candidato", "Categoria de despesa", "Valor gasto (R$)"]
    )
    print("Detalhamento por categoria vazio: Top 10 ou chave de ligação indisponível.")
    display(gastos_por_categoria)


## 10. Exportar as tabelas para Excel

In [ ]:
arquivo_excel = "analise_gastos_eleitorais.xlsx"

with pd.ExcelWriter(arquivo_excel) as writer:
    tabela_geral.to_excel(
        writer, sheet_name="Todos os candidatos", index=False
    )
    top10.to_excel(
        writer, sheet_name="Top 10 gastos", index=False
    )
    gastos_por_categoria.to_excel(
        writer, sheet_name="Despesas por categoria", index=False
    )

print("Planilha criada:", arquivo_excel)

# Se estiver executando no Google Colab, inicia o download.
try:
    from google.colab import files
    files.download(arquivo_excel)
except ImportError:
    print("Ambiente fora do Colab. O arquivo ficou salvo na pasta atual.")


## 11. Cuidados antes de apresentar os resultados

- Confira se todos os três anos foram processados sem erros.
- Confira se o CSV selecionado para despesas é realmente o de despesas de candidatos.
- Confira se o arquivo de resultados contém a situação final do turno, e não apenas resultados parciais.
- Em eleições com segundo turno, a situação de eleito deve refletir o resultado final.
- A tabela Top 10 é separada por ano e cargo; assim, não mistura candidatos a presidente com candidatos a governador.
- A seção 9 detalha as categorias de despesas dos 10 maiores gastadores.
- Despesa declarada não é a mesma coisa que dinheiro arrecadado. Aqui, o indicador é a soma de `VR_DESPESA_CONTRATADA`, caso essa coluna exista no arquivo selecionado.
